In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import osn
import time

In [ ]:
# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [ ]:
# PARAMETERS ------------------------------------------------------------------------------------

FIRE_MASK_MIN   = 8     # >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # 5% cumulative detections = onset
END_THRESHOLD   = 0.95  # 95% cumulative detections = end of season
MIN_DETECTIONS  = 20    # minimum annual detections to compute metrics

In [ ]:
# Define region ------------------------------------------------------------------------------------

izmir = ee.Geometry.Rectangle([26.5, 37.8, 28.5, 39.0])
eco_geometry = izmir
ECO_NAME = "Izmir"

In [ ]:
# LOAD TERRA + AQUA collections --------------------------------------------------------------------

terra = ee.ImageCollection("MODIS/061/MOD14A1") \
          .select('FireMask') \
          .filterBounds(eco_geometry)

aqua  = ee.ImageCollection("MODIS/061/MYD14A1") \
          .select('FireMask') \
          .filterBounds(eco_geometry)

print('Terra image count:', terra.size().getInfo())
print('Aqua image count:', aqua.size().getInfo())

In [ ]:
# Function: get daily counts — single server-side call per year ------------------------------------

def get_daily_counts(eco_geometry, year):

    start      = ee.Date.fromYMD(year, 1, 1)
    end        = ee.Date.fromYMD(year + 1, 1, 1)

    # Pre-filter both collections to this year
    terra_year = terra.filterDate(start, end)
    aqua_year  = aqua.filterDate(start, end)

    empty = ee.Image.constant(0).rename('FireMask').toUint8()

    # Build a server-side list of day offsets: 0, 1, 2, ... 364
    n_days  = 366 if ee.Date(start).advance(366, 'day').get('year').getInfo() == year + 1 else 365
    day_seq = ee.List.sequence(0, n_days - 1)

    def make_daily_image(d):
        d        = ee.Number(d)
        date     = start.advance(d, 'day')
        date_end = date.advance(1, 'day')

        terra_day = terra_year.filterDate(date, date_end)
        aqua_day  = aqua_year.filterDate(date, date_end)

        t = ee.Image(ee.Algorithms.If(
            terra_day.size().gt(0),
            terra_day.select('FireMask').max(),
            empty
        ))
        a = ee.Image(ee.Algorithms.If(
            aqua_day.size().gt(0),
            aqua_day.select('FireMask').max(),
            empty
        ))

        combined    = t.max(a)
        fire_binary = combined.gte(FIRE_MASK_MIN).unmask(0).rename('fire')

        # Store the count as an image property
        count = fire_binary.reduceRegion(
            reducer   = ee.Reducer.sum(),
            geometry  = eco_geometry,
            scale     = 1000,
            maxPixels = 1e8
        ).get('fire')

        # Return a 1-pixel constant image with the count stored as a property
        return ee.Image.constant(0).set('doy', d.add(1)).set('count', count)

    # Map over all days server-side — no Python loop
    daily_collection = ee.ImageCollection(day_seq.map(make_daily_image))

    # Fetch DOYs and counts in TWO calls (one per property array)
    doys   = daily_collection.aggregate_array('doy').getInfo()
    counts = daily_collection.aggregate_array('count').getInfo()

    return pd.DataFrame({
        'doy'          : doys,
        'n_detections' : [int(c) if c is not None else 0 for c in counts]
    })

In [ ]:
# Run for one test year and plot -------------------------------------------------------------------
TEST_YEAR = 2008

df_2008 = get_daily_counts(eco_geometry, TEST_YEAR)

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(df_2008['doy'], df_2008['n_detections'], color='firebrick', width=1)
plt.xlabel('Day of Year')
plt.ylabel('Fire Detections')
plt.title(f'Daily Fire Activity — {ECO_NAME}, {TEST_YEAR}')
plt.tight_layout()
plt.show()

print('Total detections:', df_2008['n_detections'].sum())
print('Peak DOY:', df_2008.loc[df_2008['n_detections'].idxmax(), 'doy'])

In [ ]:
# Function: compute timing metrics from daily counts -----------------------------------------------
def compute_timing_metrics(df, year):
    """
    Given a daily counts DataFrame for one ecoregion-year,
    returns a dict with onset_doy, peak_doy, end_doy, season_length.
    Returns None if total detections fall below MIN_DETECTIONS.
    """
    total = df['n_detections'].sum()

    if total < MIN_DETECTIONS:
        print(f'{year}: insufficient detections ({total}), skipping.')
        return None

    df = df.copy().sort_values('doy')
    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    # Onset: first doy where cumulative fraction crosses 5%
    onset_rows = df[cum_frac >= ONSET_THRESHOLD]
    # End: first doy where cumulative fraction crosses 95%
    end_rows   = df[cum_frac >= END_THRESHOLD]

    if onset_rows.empty or end_rows.empty:
        print(f'{year}: could not compute onset or end, skipping.')
        return None

    # Peak: doy of maximum 7-day rolling mean
    df['rolling'] = df['n_detections'].rolling(7, center=True, min_periods=1).mean()
    peak_doy      = int(df.loc[df['rolling'].idxmax(), 'doy'])

    onset_doy  = int(onset_rows.iloc[0]['doy'])
    end_doy    = int(end_rows.iloc[0]['doy'])
    length     = end_doy - onset_doy + 1

    return {
        'year'         : year,
        'onset_doy'    : onset_doy,
        'peak_doy'     : peak_doy,
        'end_doy'      : end_doy,
        'season_length': length,
        'n_detections' : int(total)
    }

In [ ]:
# Test timing metrics on 2008 ----------------------------------------------------------------------
metrics_2008 = compute_timing_metrics(df_2008, TEST_YEAR)
print(metrics_2008)

In [ ]:
# Run all years and collect results ----------------------------------------------------------------
YEARS = list(range(2003, 2025))

all_daily = {}
all_metrics = []

for i, year in enumerate(YEARS):
    t0 = time.time()
    print(f'Processing {year}... ({i+1}/{len(YEARS)})')
    
    df_year = get_daily_counts(eco_geometry, year)
    all_daily[year] = df_year
    metrics = compute_timing_metrics(df_year, year)
    if metrics is not None:
        all_metrics.append(metrics)
    
    elapsed = time.time() - t0
    print(f'  → done in {elapsed:.1f}s')

metrics_df = pd.DataFrame(all_metrics)
print(metrics_df)

In [ ]:
# Save outputs to Google Drive (or local path)
output_dir = 'outputs/izmir'
os.makedirs(output_dir, exist_ok=True)

# Save the summary metrics table
metrics_df.to_csv(f'{output_dir}/metrics_izmir.csv', index=False)

# Save each year's daily counts as a separate CSV
for year, df_year in all_daily.items():
    df_year.to_csv(f'{output_dir}/daily_{year}_izmir.csv', index=False)

print('Saved metrics and daily CSVs to', output_dir)

In [ ]:
# Plot timing metrics across years
metrics_df['year'] = metrics_df['year'].astype(int)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f'Fire Season Timing —', fontsize=13)

axes[0, 0].plot(metrics_df['year'], metrics_df['onset_doy'],     marker='o', color='orange')
axes[0, 0].set_title('Onset DOY')

axes[0, 1].plot(metrics_df['year'], metrics_df['peak_doy'],      marker='o', color='firebrick')
axes[0, 1].set_title('Peak DOY')

axes[1, 0].plot(metrics_df['year'], metrics_df['end_doy'],       marker='o', color='steelblue')
axes[1, 0].set_title('End DOY')

axes[1, 1].plot(metrics_df['year'], metrics_df['season_length'], marker='o', color='green')
axes[1, 1].set_title('Season Length (days)')

for ax in axes.flatten():
    ax.set_xlabel('Year')
    ax.set_xticks(metrics_df['year'])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()